# Medical Diagnosis Support using Decision Trees
## Breast Cancer Wisconsin (Diagnostic) Dataset

**MDI3003 - Advanced Predictive Analytics - Lab 02**

### Objective

Develop an interpretable machine-learning pipeline that classifies breast
tumours as **Malignant** or **Benign** from Decision-Tree-based models, while
following industry-standard predictive-analytics practice. The project
emphasises data-quality assessment, leakage-free preprocessing, explainable AI,
reproducibility, hyperparameter optimisation, clinical error analysis and
responsible AI.

### Educational disclaimer

This notebook is developed solely for educational and research purposes. The
resulting models are **NOT** clinically validated and must **NOT** be used for
diagnosis, treatment planning, patient triage, or any real-world healthcare
decision. The predictions only reproduce patterns learned from a public
benchmark dataset.

# 1. Business Understanding and Problem Framing

## Problem statement

Early detection of breast cancer substantially improves survival rates. This
notebook builds an interpretable binary classifier that predicts whether a
tumour is malignant or benign, using numerical descriptors extracted from
digitised fine-needle-aspiration (FNA) images. Unlike black-box models, Decision
Trees expose transparent if-then rules, which is useful for explainable baselines
in healthcare research.

## Prediction task

| Item | Specification |
|:---|:---|
| **Observation unit** | One FNA image measurement set per patient |
| **Target variable** | `Diagnosis` (Malignant vs Benign) |
| **Positive class** | Malignant -> encoded as **1** |
| **Negative class** | Benign -> encoded as **0** |
| **Prediction time** | All predictors assumed available before diagnosis is confirmed |
| **Intended use** | Educational predictive-analytics prototype |
| **Out of scope** | Diagnosis, treatment recommendation, clinical decision support, patient risk assessment |

# 2. Environment Setup

Import the required libraries, set display and plotting defaults, fix random
seeds for reproducibility, and record software versions. The shared engine
`meddiag_common` (`src/`) is placed on the path so the notebook, the CLI and the
GUI all use the same dataset metadata, preprocessing, metrics and persistence
code and can never disagree on protocol.

In [ ]:
# ---- 2. Environment Setup ----
# Imports, display options, random seeding for reproducibility, and a quick
# version report. A `report_figures/` dir is created for publication-quality
# 300-DPI exports used by the lab report docx.
import os
import json
import random
import warnings
from pathlib import Path
from datetime import datetime

# ==========================================
# Data Manipulation
# ==========================================

import numpy as np
import pandas as pd

# ==========================================
# Visualization
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# Scikit-Learn
# ==========================================

from sklearn.datasets import load_breast_cancer

# Display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

warnings.filterwarnings("ignore")

plt.style.use("ggplot")
sns.set_context("talk")

# ==========================================
# Reproducibility
# ==========================================

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# ==========================================
# Version Information
# ==========================================

import platform
import sklearn

print("="*60)
print("Environment Information")
print("="*60)

print(f"Python Version      : {platform.python_version()}")
print(f"Pandas Version      : {pd.__version__}")
print(f"Numpy Version       : {np.__version__}")
print(f"Scikit-Learn Version: {sklearn.__version__}")
print(f"Random State        : {RANDOM_STATE}")
# Create report figures directory
os.makedirs("report_figures", exist_ok=True)

# 3. Dataset Loading

The Breast Cancer Wisconsin (Diagnostic) dataset is shipped with scikit-learn
and is also cached under `data/breast_cancer_wisconsin.csv` (UCI id 17). It has
569 observations, 30 numerical predictors, a binary target and no missing
values - a clean core workflow for interpretable Decision-Tree modelling. For
this lab the **Malignant** tumour is the positive class (encoded 1).

In [ ]:
# ---- 3. Load the Breast Cancer Wisconsin dataset (scikit-learn) ----
# `load_breast_cancer(as_frame=True)` returns 30 numeric predictors + the target.
# X holds features, y holds the raw 0/1 target (which sklearn encodes as
# 0=Benign, 1=Malignant - i.e. Malignant is already the positive class).
raw_data = load_breast_cancer(as_frame=True)

X = raw_data.data.copy()
y = raw_data.target.copy()

In [ ]:
# ---- 3.1 Binarise the target and assemble one combined dataframe ----
# Remap so the encoded target is 1=Malignant (positive) / 0=Benign. `dataset`
# keeps features and the binarised target together for EDA convenience.
# Convert target

target_mapping = {
    0: 1,   # Malignant
    1: 0    # Benign
}

y = y.map(target_mapping)

y.name = "Diagnosis"

dataset = X.copy()
dataset["Diagnosis"] = y

dataset.head()

print("="*60)
print("Dataset Summary")
print("="*60)

print(f"Rows               : {dataset.shape[0]}")
print(f"Columns            : {dataset.shape[1]}")
print(f"Features           : {X.shape[1]}")
print(f"Target Classes     : {y.nunique()}")

print("\nTarget Mapping")
print("1 -> Malignant")
print("0 -> Benign")

print("\nClass Distribution")

display(y.value_counts())

# 4. Data Understanding and Audit

Before any modelling, audit the dataset thoroughly: structure, dtypes,
statistical profile, target distribution, missingness, duplicates,
constant/quasi-constant features, per-feature distributions, outliers,
correlation, multicollinearity and feature-target relationships. The goal is to
surface data-quality issues and to guide - not over-engineer - preprocessing
while keeping the workflow leakage-safe.

In [ ]:
# ---- 4.1 Dataset shape overview ----
print("="*70)
print("Dataset Shape")
print("="*70)

print(f"Rows      : {dataset.shape[0]}")
print(f"Columns   : {dataset.shape[1]}")
print(f"Features  : {X.shape[1]}")
print(f"Target    : Diagnosis")

In [ ]:
# ---- 4. Peek at the first rows ----
dataset.head()

In [ ]:
# ---- 4. Peek at a random sample (seed-locked) for a sanity check ----
dataset.sample(5, random_state=RANDOM_STATE)

## 4.1 First look

The dataset contains only numerical predictor variables; each row is one patient
sample; the target is binary; no identifier columns are present.

In [ ]:
# ---- 4.2 dtypes via info() ----
dataset.info()

In [ ]:
# ---- 4.2 Build a compact dtype table for the report ----
dtype_table = (
    dataset
    .dtypes
    .reset_index()
)

dtype_table.columns = ["Feature","Data Type"]

display(dtype_table)

## 4.2 Data types

All predictors are continuous numerical measurements. No categorical, datetime
or identifier features exist, so no categorical encoding is required.

In [ ]:
# ---- 4.3 Statistical profile: extend describe() with median/variance/skew/kurtosis ----
summary = dataset.describe().T

summary
summary["median"] = dataset.median()

summary["variance"] = dataset.var()

summary["skewness"] = dataset.skew()

summary["kurtosis"] = dataset.kurt()

summary

## 4.3 Statistical profile

Summary statistics reveal substantial variation in feature ranges and noticeable
skewness/kurtosis, so distributions are not Gaussian. Decision-Tree algorithms
are robust to non-normality, so no transformation is required at this stage.

In [ ]:
# ---- 4.4 Target distribution counts + percentages ----
target_counts = y.value_counts().sort_index()

target_percent = (
    y.value_counts(normalize=True)
      .sort_index()*100
)

display(pd.DataFrame({
    "Count":target_counts,
    "Percentage":target_percent.round(2)
}))

In [ ]:
# ---- 4.4 Plot the target class distribution (also saved for the report) ----
plt.figure(figsize=(7,5))

ax = sns.countplot(
    x=y,
    palette="Set2"
)

for container in ax.containers:
    ax.bar_label(container)

plt.xticks([0,1],["Benign","Malignant"])

plt.title("Target Class Distribution")

plt.xlabel("Diagnosis")

plt.ylabel("Count")

plt.tight_layout()

plt.savefig("report_figures/fig1_target_dist_bc.png", dpi=300, bbox_inches="tight")
plt.show()

## 4.4 Target distribution

The class distribution is only mildly imbalanced (Benign more frequent than
Malignant), so SMOTE-style resampling is unnecessary; class weighting will still
be evaluated during model development.

In [ ]:
# ---- 4.5 Missing-value audit (count + percentage) ----
missing = pd.DataFrame({

    "Missing Values":dataset.isnull().sum(),

    "Percentage":(
        dataset.isnull().mean()*100
    )

})

missing.sort_values(
    by="Missing Values",
    ascending=False
).head()

## 4.5 Missing values

No missing values are present. The modelling pipeline will still include an
imputer to keep the workflow reusable on datasets that do contain missingness.

In [ ]:
# ---- 4.6 Duplicate-row audit ----
duplicates = dataset.duplicated().sum()

print(f"Duplicate Rows : {duplicates}")
if duplicates>0:
    display(dataset[dataset.duplicated()])

## 4.6 Duplicates

No duplicate observations are detected, so no de-duplication is required.

In [ ]:
# ---- 4.7 Constant-feature audit (zero-variance columns) ----
constant = [
    c
    for c in X.columns
    if X[c].nunique()==1
]

constant

In [ ]:
# ---- 4.7 Quasi-constant audit (one value dominates > 99%) ----
quasi = []

for col in X.columns:

    dominant = X[col].value_counts(normalize=True).iloc[0]

    if dominant>0.99:

        quasi.append(col)

quasi

## 4.7 Constant / quasi-constant features

No constant or quasi-constant features are present; all variables carry
information for the predictive task.

In [ ]:
# ---- 4.8 Helper: per-feature distributions by class ----
# Grid of histplots with KDE, density-normalised per class so class imbalance
# does not distort the visual comparison.
def plot_feature_distributions(df, cols, target):

    n_cols = 3

    n_rows = int(np.ceil(len(cols)/n_cols))

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(18,5*n_rows)
    )

    axes = axes.flatten()

    for ax,col in zip(axes,cols):

        sns.histplot(

            data=df,

            x=col,

            hue=target,

            kde=True,

            stat="density",

            common_norm=False,

            ax=ax

        )

        ax.set_title(col)

    for ax in axes[len(cols):]:

        ax.remove()

    plt.tight_layout()

    plt.show()

In [ ]:
# ---- 4.8 Render per-feature distributions for all 30 predictors ----
plot_feature_distributions(

    dataset,

    X.columns,

    "Diagnosis"

)

## 4.8 Feature distributions by class

Several features show clear separation between malignant and benign tumours
(e.g. worst radius, worst perimeter, worst concavity, mean concavity, mean
radius). These are likely to be selected near the top of the tree.

In [ ]:
# ---- 4.9 Outlier check: per-feature boxplots ----
fig, axes = plt.subplots(10,3,figsize=(18,40))

axes = axes.flatten()

for ax,col in zip(axes,X.columns):

    sns.boxplot(

        y=dataset[col],

        ax=ax

    )

    ax.set_title(col)

for ax in axes[len(X.columns):]:

    ax.remove()

plt.tight_layout()

plt.show()

## 4.9 Outliers

Several numerical variables contain apparent outliers. Decision Trees partition
observations using thresholds rather than distances, so they are considerably
less outlier-sensitive than linear or nearest-neighbour models; no outlier
removal is performed.

In [ ]:
# ---- 4.10 Distribution-shape audit: skew + kurtosis ----
shape = pd.DataFrame({

    "Skewness":X.skew(),

    "Kurtosis":X.kurt()

})

shape.sort_values(
    by="Skewness",
    ascending=False
)

## 4.10 Distribution shape

Many variables are positively skewed. Decision Trees are invariant to monotonic
transformations and make no Gaussian assumption, so no log or Box-Cox
transformation is required.

# 5. Advanced Exploratory Data Analysis

Investigate predictor relationships, multicollinearity, feature relevance and
potential data leakage before model development. These analyses guide feature
selection, preprocessing and interpretation while keeping the workflow
leakage-safe.

In [ ]:
# ---- 5.1 Correlation matrix across all 30 numeric predictors ----
# Lower-triangular heatmap; upper triangle is masked to reduce clutter.
corr_matrix = X.corr()

plt.figure(figsize=(18,15))

mask = np.triu(np.ones_like(corr_matrix))

sns.heatmap(
    corr_matrix,
    mask=mask,
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.4,
    cbar_kws={"shrink":0.8}
)

plt.title("Correlation Matrix of Numerical Features", fontsize=18)

plt.tight_layout()

plt.savefig("report_figures/fig2_correlation_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

## 5.1 Correlation matrix

The correlation matrix exposes several highly correlated feature groups. This is
expected because multiple measurements are derived from the same tumour
characteristics (radius, perimeter, area, texture, ...). Tree models are robust
to multicollinearity, so no feature is removed on correlation grounds alone.

In [ ]:
# ---- 5.2 Tabulate the highly-correlated pairs (|r| >= 0.85) ----
corr = X.corr().abs()

upper = corr.where(
    np.triu(np.ones(corr.shape),k=1).astype(bool)
)

high_corr = (
    upper.stack()
         .reset_index()
)

high_corr.columns = ["Feature A","Feature B","Correlation"]

high_corr = high_corr[
    high_corr["Correlation"]>=0.85
].sort_values(
    by="Correlation",
    ascending=False
)

display(high_corr)

## 5.2 Highly correlated pairs

Several pairs exceed 0.90 absolute correlation and capture closely related
physical characteristics (overlapping information). Tree-based models select
informative split variables automatically, so feature elimination is
unnecessary.

In [ ]:
# ---- 5.3 Feature-target correlation (point-biserial with the binary target) ----
# Sorted by absolute value to surface the strongest linear associations.
corr_target = dataset.corr()["Diagnosis"][:-1]

corr_target = (
    corr_target
    .sort_values(
        key=abs,
        ascending=False
    )
)

corr_target

In [ ]:
# ---- 5.3 Bar plot of feature-target correlations (saved for the report) ----
plt.figure(figsize=(10,9))

corr_target.sort_values().plot.barh()

plt.title("Feature Correlation with Target")

plt.xlabel("Correlation")

plt.tight_layout()

plt.savefig("report_figures/fig3_feature_corr_bc.png", dpi=300, bbox_inches="tight")
plt.show()

## 5.3 Feature-target correlation

Features such as worst radius, worst perimeter, worst concavity, mean concavity
and mean perimeter have the strongest linear association with the diagnosis.
Correlation alone does not decide model importance, but these variables are
expected to appear near the top of the tree.

In [ ]:
# ---- 5.4 Pairwise scatter of the top-5 features, coloured by class ----
# Corner pairplot keeps only the lower triangle (no redundant upper panels).
top_features = corr_target.abs().head(5).index.tolist()

pair_df = dataset[top_features + ["Diagnosis"]]

sns.pairplot(

    pair_df,

    hue="Diagnosis",

    diag_kind="kde",

    corner=True,

    plot_kws={"alpha":0.7}

)

plt.show()

## 5.4 Pairwise relationships

The pairwise plots show substantial separation between benign and malignant
samples; some feature combinations are almost linearly separable, so relatively
shallow trees may already perform well.

In [ ]:
# ---- 5.5 Variance Inflation Factor audit (multicollinearity) ----
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif = pd.DataFrame()

vif["Feature"] = X.columns

vif["VIF"] = [

    variance_inflation_factor(

        X.values,

        i

    )

    for i in range(X.shape[1])

]

vif.sort_values(

    by="VIF",

    ascending=False

).head(15)

## 5.5 Variance inflation factors

Several variables have extremely high VIF, confirming strong multicollinearity -
expected because many features encode related geometric measurements from the
same image. Decision Trees are far less affected than linear regression, so no
corrective action is needed.

In [ ]:
# ---- 5.6 Feature variance audit + bar plot of the top-15 ----
feature_variance = X.var().sort_values(
    ascending=False
)

feature_variance.head(15)
plt.figure(figsize=(10,8))

feature_variance.head(15).sort_values().plot.barh()

plt.title("Highest Variance Features")

plt.tight_layout()

plt.show()

## 5.6 Feature variance

Variance differs considerably across variables. Decision Trees are invariant to
monotonic scaling, so high variance is not a problem and no standardisation is
required.

In [ ]:
# ---- 5.7 Consolidated leakage / quality audit table ----
audit = pd.DataFrame({

    "Missing":X.isna().sum(),

    "Unique":X.nunique(),

    "Correlation with Target":corr_target,

    "Potential Leakage":"No"

})

audit.head()

In [ ]:
# ---- 5.7 Printable leakage-audit summary ----
print("="*60)

print("Leakage Audit")

print("="*60)

print("Identifier Columns :",0)

print("Duplicate Rows     :",dataset.duplicated().sum())

print("Missing Values     :",dataset.isna().sum().sum())

print("Target Leakage     : None Identified")

## 5.7 Leakage audit

The dataset contains no identifiers, timestamps, post-diagnosis variables or
outcome-derived attributes. All predictors are assumed available before the
prediction task, so the data is suitable for supervised learning without target
leakage.

In [ ]:
# ---- 5.8 Hierarchical clustering of predictors by correlation ----
# `clustermap` reorders features by similarity, exposing correlated blocks.
sns.clustermap(

    corr_matrix,

    figsize=(14,14),

    cmap="coolwarm",

    center=0

)

plt.show()

## 5.8 Hierarchical feature clustering

Hierarchical clustering reveals coherent groups of related measurements,
particularly among radius, perimeter, area and concavity variables - additional
evidence that several features describe the same biological property.

In [ ]:
# ---- 5.9 One-line EDA summary table for the report ----
eda_summary = pd.DataFrame({

    "Finding":[

        "Missing Values",

        "Duplicate Rows",

        "Constant Features",

        "Categorical Variables",

        "Target Leakage",

        "Class Imbalance",

        "Strong Correlation",

        "Outliers",

        "Scaling Required"

    ],

    "Result":[

        "None",

        dataset.duplicated().sum(),

        len(constant),

        "None",

        "None Detected",

        "Mild",

        "Present",

        "Present",

        "No"

    ]

})

eda_summary

# 6. Data Preparation

The EDA showed the data is fully numeric with no missingness, duplicates or
categorical features. Preprocessing is therefore minimal, but a full scikit-learn
pipeline is still constructed to guarantee reproducibility, leakage prevention,
compatibility with future datasets and production-ready deployment. The test set
is created **before** any model training or tuning and remains untouched until
the final evaluation.

In [ ]:
# ---- 6.1 Lock the stratified 80/20 train/test split before any tuning ----
# `stratify=y` preserves the class ratio in both subsets; the test set is now
# frozen and is only touched once at the final evaluation.
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

TEST_SIZE = 0.20

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

print("="*60)
print("Dataset Split")
print("="*60)

print(f"Training Samples : {X_train.shape[0]}")
print(f"Testing Samples  : {X_test.shape[0]}")

print(f"Training Features: {X_train.shape[1]}")

In [ ]:
# ---- 6.1 Split summary: rows and positive-class prevalence per subset ----
split_summary = pd.DataFrame({

    "Dataset":[
        "Training",
        "Testing"
    ],

    "Rows":[
        len(X_train),
        len(X_test)
    ],

    "Positive %":[
        y_train.mean()*100,
        y_test.mean()*100
    ]

})

split_summary

## 6.1 Stratified split

A stratified 80/20 split preserves the malignant/benign ratio in both subsets.
The test set is now locked and will not be used during model development or
hyperparameter optimisation.

In [ ]:
# ---- 6.2 Cross-validation strategy: 5-fold stratified, shuffled, seed-locked ----
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

print(cv)

## 6.2 Cross-validation strategy

Stratified K-Fold (5 folds, shuffled, seed 42) keeps approximately the same
class distribution in every fold. This reduces metric variability and is the
preferred validation strategy for binary problems with mild imbalance.

In [ ]:
# ---- 6.3 Record the numeric feature list for the preprocessing pipeline ----
numeric_features = X.columns.tolist()

print(f"Numerical Features : {len(numeric_features)}")

In [ ]:
# ---- 6.3 Build a leakage-safe preprocessor: median imputer only ----
# No scaling (trees are scale-invariant) and no encoding (all features numeric).
# Wrapped as a ColumnTransformer so the imputer is fit inside CV folds.
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

numeric_pipeline = Pipeline(

    steps=[

        (
            "imputer",

            SimpleImputer(strategy="median")

        )

    ]

)

preprocessor = ColumnTransformer(

    transformers=[

        (

            "num",

            numeric_pipeline,

            numeric_features

        )

    ],

    remainder="drop"

)

preprocessor

## 6.3 Why an imputer with no missing values?

The dataset has no missingness, but embedding an imputation step in the pipeline
improves reproducibility and lets the workflow be reused on datasets with
missing values without code changes.

In [ ]:
# ---- 6.4 Cross-validation scoring dictionary ----
# All metrics used for CV selection are declared here once and reused.
SCORING = {

    "Accuracy":"accuracy",

    "Balanced Accuracy":"balanced_accuracy",

    "Precision":"precision",

    "Recall":"recall",

    "F1":"f1",

    "ROC-AUC":"roc_auc",

    "PR-AUC":"average_precision"

}

In [ ]:
# ---- 6.4 Import the metric callables used by evaluate_model / cv_report ----
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    matthews_corrcoef,
    brier_score_loss,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

In [ ]:
# ---- 6.5 Reusable on-dataset metric reporter ----
# Returns a Series of accuracy, balanced accuracy, precision, recall,
# specificity, F1, ROC-AUC and PR-AUC for a fitted model on (X, y).
def evaluate_model(model, X, y):

    pred = model.predict(X)

    prob = model.predict_proba(X)[:,1]

    tn, fp, fn, tp = confusion_matrix(
        y,
        pred
    ).ravel()

    specificity = tn/(tn+fp)

    metrics = {

        "Accuracy":
            accuracy_score(y,pred),

        "Balanced Accuracy":
            balanced_accuracy_score(y,pred),

        "Precision":
            precision_score(y,pred),

        "Recall":
            recall_score(y,pred),

        "Specificity":
            specificity,

        "F1":
            f1_score(y,pred),

        "ROC-AUC":
            roc_auc_score(y,prob),

        "PR-AUC":
            average_precision_score(y,prob)

    }

    return pd.Series(metrics)

In [ ]:
# ---- 6.5 Reusable CV reporter ----
# `cv_report` runs `cross_validate` and returns a tidy summary table +
# a one-row comparison dict (train/validation mean+std per metric).
from sklearn.model_selection import cross_validate
def cv_report(
    model,
    model_name,
    X,
    y,
    cv,
    scoring,
    return_predictions=False
):
    """
    Performs cross validation and returns:
    - complete CV dataframe
    - summary dataframe
    - model comparison row
    """

    cv_results = cross_validate(

        estimator=model,

        X=X,

        y=y,

        cv=cv,

        scoring=scoring,

        return_train_score=True,

        n_jobs=-1

    )

    cv_df = pd.DataFrame(cv_results)

    summary = pd.DataFrame(index=scoring.keys())

    for metric in scoring.keys():

        summary.loc[metric, "Train Mean"] = cv_df[f"train_{metric}"].mean()
        summary.loc[metric, "Train Std"] = cv_df[f"train_{metric}"].std()

        summary.loc[metric, "Validation Mean"] = cv_df[f"test_{metric}"].mean()
        summary.loc[metric, "Validation Std"] = cv_df[f"test_{metric}"].std()

    comparison_row = {

        "Model": model_name,

        "Accuracy":
            cv_df["test_Accuracy"].mean(),

        "Balanced Accuracy":
            cv_df["test_Balanced Accuracy"].mean(),

        "Precision":
            cv_df["test_Precision"].mean(),

        "Recall":
            cv_df["test_Recall"].mean(),

        "F1":
            cv_df["test_F1"].mean(),

        "ROC-AUC":
            cv_df["test_ROC-AUC"].mean(),

        "PR-AUC":
            cv_df["test_PR-AUC"].mean()

    }

    if return_predictions:

        return cv_df, summary.round(4), comparison_row

    return summary.round(4), comparison_row

In [ ]:
# ---- 6.5 Initialise the empty model-comparison accumulator ----
model_comparison = []

## 6.4 Data-preparation summary

Preprocessing is finalised: a locked test set, stratified split, 5-fold
stratified CV, pipeline-based preprocessing, reusable evaluation helpers and a
fixed random seed. The notebook is ready for baseline model development.

# 7. Baseline Model Development

Before complex models, establish baselines so we can verify that machine learning
actually beats the trivial rules. Two baselines are built: a Dummy classifier
that ignores the features and predicts the prior class distribution, and a basic
unconstrained CART that learns if-then rules from the predictors. The Dummy sets
the floor; the basic CART reveals whether unconstrained trees overfit.

In [ ]:
# ---- 7.0 Pipeline helper: preprocessor + classifier ----
# Every model is wrapped the same way so preprocessing is fit inside each CV
# fold and never leaks information from validation into training.
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.model_selection import cross_validate

In [ ]:
# ---- 7.0 build_pipeline() definition ----
def build_pipeline(estimator):

    return Pipeline(

        steps=[

            ("preprocessor", preprocessor),

            ("classifier", estimator)

        ]

    )

In [ ]:
# ---- 7.1 Dummy baseline (predicts the majority class prior) ----
dummy_pipeline = build_pipeline(

    DummyClassifier(
        strategy="prior"
    )

)

In [ ]:
# ---- 7.1 Cross-validate the Dummy and append its row to the comparison ----
dummy_summary, dummy_row = cv_report(

    dummy_pipeline,

    "Dummy Baseline",

    X_train,

    y_train,

    cv,

    SCORING

)

model_comparison.append(dummy_row)

dummy_summary

## 7.1 Dummy baseline observation

Because the Dummy ignores the features, any meaningful model should substantially
outperform it across all metrics - the Dummy sets the minimum expected
performance for this task.

In [ ]:
# ---- 7.2 Basic unconstrained CART (gini, no depth/leaf limits) ----
basic_cart = build_pipeline(

    DecisionTreeClassifier(

        criterion="gini",

        random_state=RANDOM_STATE

    )

)

In [ ]:
# ---- 7.2 Cross-validate the basic CART and append its row ----
cart_summary, cart_row = cv_report(

    basic_cart,

    "Basic CART",

    X_train,

    y_train,

    cv,

    SCORING

)

model_comparison.append(cart_row)

cart_summary

## 7.2 Basic CART observation

The unconstrained tree is perfect on training folds but weaker on validation
folds - the classic overfitting signature. This motivates hyperparameter tuning
and pruning in the following sections.

In [ ]:
# ---- 7.2 Fit the basic CART on the train split to inspect its structure ----
basic_cart.fit(

    X_train,

    y_train

)

In [ ]:
# ---- 7.3 Report tree complexity (depth / leaves / total nodes) ----
tree = basic_cart.named_steps["classifier"]

print("="*60)

print("Basic CART Statistics")

print("="*60)

print(f"Tree Depth     : {tree.get_depth()}")

print(f"Leaf Nodes     : {tree.get_n_leaves()}")

print(f"Total Nodes    : {tree.tree_.node_count}")

## 7.3 Tree complexity

Tree complexity is measured by depth, number of leaves and total nodes; very deep
trees with many leaves indicate high variance and an increased overfitting risk.

In [ ]:
# ---- 7.3 Visualise the top 3 levels of the basic CART ----
from sklearn.tree import plot_tree
plt.figure(figsize=(20,10))

plot_tree(

    tree,

    feature_names=X.columns,

    class_names=[

        "Benign",

        "Malignant"

    ],

    filled=True,

    rounded=True,

    proportion=True,

    precision=2,

    max_depth=3

)

plt.title("Basic CART (First Three Levels)")

plt.tight_layout()

plt.show()

In [ ]:
# ---- 7.3 Export the basic CART's full if-then rules as text (truncated) ----
from sklearn.tree import export_text
rules = export_text(

    tree,

    feature_names=list(X.columns)

)

print(rules[:5000])

In [ ]:
# ---- 7.4 First comparison table: Dummy vs Basic CART (CV ROC-AUC sorted) ----
comparison_df = (

    pd.DataFrame(model_comparison)

      .sort_values(

          by="ROC-AUC",

          ascending=False

      )

)

comparison_df.round(4)

# 8. Advanced Tree Models

Define reusable tuning helpers, then tune a CART over pre-pruning
hyperparameters, select `ccp_alpha` through cost-complexity pruning, and finally
tune a Random Forest. Each model is selected using training-only CV; the locked
test set is untouched.

In [ ]:
# ---- 8.0 Generic tuner supporting GridSearchCV or RandomizedSearchCV ----
# Returns the best fitted estimator, the best params, the cv_results_ dataframe
# and the best CV score for the refit metric.
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

def tune_model(pipeline, param_grid, X, y, cv, scoring, refit='roc_auc', 
               n_jobs=-1, return_train_score=True, search_type='grid'):
    """
    Perform hyperparameter tuning using GridSearchCV or RandomizedSearchCV.
    
    Parameters:
        pipeline: sklearn Pipeline or estimator
        param_grid: dict of parameters to search
        X, y: training data
        cv: cross-validation object
        scoring: dict of scoring metrics
        refit: metric to select best model
        n_jobs: parallel jobs
        return_train_score: bool
        search_type: 'grid' or 'random'
    
    Returns:
        best_estimator, best_params, cv_results_df, best_score
    """
    if search_type == 'grid':
        search = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            scoring=scoring,
            refit=refit,
            cv=cv,
            n_jobs=n_jobs,
            return_train_score=return_train_score,
            verbose=1
        )
    else:
        search = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=param_grid,
            scoring=scoring,
            refit=refit,
            cv=cv,
            n_jobs=n_jobs,
            return_train_score=return_train_score,
            n_iter=50,  # default for random
            verbose=1,
            random_state=RANDOM_STATE
        )
    search.fit(X, y)
    cv_results = pd.DataFrame(search.cv_results_)
    return search.best_estimator_, search.best_params_, cv_results, search.best_score_

In [ ]:
# ---- 8.1 Tune a CART over pre-pruning hyperparameters (training-only CV) ----
# Grid over criterion / max_depth / min_samples_split / min_samples_leaf /
# class_weight, refit on ROC-AUC.
# Build a pipeline with the imputer and a DecisionTreeClassifier
cart_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', DecisionTreeClassifier(random_state=RANDOM_STATE))
])

# Define search space for pre-pruning
param_grid_cart = {
    'model__criterion': ['gini', 'entropy'],
    'model__max_depth': [3, 4, 5, 6, 8, 10, None],
    'model__min_samples_split': [2, 5, 10, 20],
    'model__min_samples_leaf': [1, 2, 5, 10],
    'model__class_weight': [None, 'balanced']
}

print("Tuning CART pre-pruning hyperparameters...")
best_cart_estimator, best_cart_params, cart_cv_results, best_cart_score = tune_model(
    pipeline=cart_pipe,
    param_grid=param_grid_cart,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=SCORING,
    refit='ROC-AUC',
    search_type='grid'
)

print("\nBest pre-pruned CART parameters:")
print(best_cart_params)
print(f"Best cv ROC-AUC: {best_cart_score:.4f}")

## 8.1 Pre-pruned CART result

Grid search picks the combination of criterion, max_depth, min_samples_split,
min_samples_leaf and class_weight that maximises cross-validated ROC-AUC. This
pre-pruned tree already reduces overfitting compared with the unconstrained
CART.

# 9. Cost-Complexity Pruning for CART

Fix the best pre-pruning parameters found above and select the optimal
`ccp_alpha` by cross-validation. Two-stage selection (pre-prune, then prune)
follows the lab manual and yields the "tuned and pruned CART".

In [ ]:
# ---- 9.1 Cost-complexity pruning: derive the alpha path from the best tree ----
# Fit a base tree with the best pre-prune parameters, compute the
# `cost_complexity_pruning_path`, restrict to a 20-point alpha grid (drop the
# trivial one-node alpha), then grid-search `ccp_alpha` via CV.
# First, train a base tree on the full training data using the best pre-pruning params
# (without ccp_alpha) to obtain the pruning path.
base_tree = DecisionTreeClassifier(
    criterion=best_cart_params['model__criterion'],
    max_depth=best_cart_params['model__max_depth'],
    min_samples_split=best_cart_params['model__min_samples_split'],
    min_samples_leaf=best_cart_params['model__min_samples_leaf'],
    class_weight=best_cart_params['model__class_weight'],
    random_state=RANDOM_STATE
)
base_tree.fit(X_train, y_train)

# Compute cost-complexity pruning path
path = base_tree.cost_complexity_pruning_path(X_train, y_train)

ccp_alphas = path["ccp_alphas"]
impurities = path["impurities"]
# Remove the trivial tree (alpha = 0) and large alpha values that yield single node
ccp_alphas = np.asarray(ccp_alphas, dtype=float)
ccp_alphas = ccp_alphas[ccp_alphas > 0]

if len(ccp_alphas) == 0:
    ccp_alphas = np.array([0.0], dtype=float)

# We'll create a pipeline with the same pre-processing and set model with best params,
# then tune ccp_alpha
prune_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', DecisionTreeClassifier(
        criterion=best_cart_params['model__criterion'],
        max_depth=best_cart_params['model__max_depth'],
        min_samples_split=best_cart_params['model__min_samples_split'],
        min_samples_leaf=best_cart_params['model__min_samples_leaf'],
        class_weight=best_cart_params['model__class_weight'],
        random_state=RANDOM_STATE
    ))
])

# Use a reduced set of alphas to speed up (e.g., take 20 evenly spaced)
if len(ccp_alphas) > 20:
    alpha_grid = np.linspace(ccp_alphas.min(), ccp_alphas.max(), 20)
else:
    alpha_grid = ccp_alphas

param_grid_alpha = {
    'model__ccp_alpha': alpha_grid
}

print("Tuning ccp_alpha via cross-validation...")
best_pruned_estimator, best_alpha_params, alpha_cv_results, best_alpha_score = tune_model(
    pipeline=prune_pipe,
    param_grid=param_grid_alpha,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=SCORING,
    refit='ROC-AUC',
    search_type='grid'
)

best_ccp_alpha = best_alpha_params['model__ccp_alpha']
print(f"\nBest ccp_alpha: {best_ccp_alpha:.6f}")
print(f"Best cv ROC-AUC after pruning: {best_alpha_score:.4f}")

# The best_pruned_estimator now contains the tuned and pruned CART
pruned_cart = best_pruned_estimator

In [ ]:
# ---- 9.2 Plot CV ROC-AUC vs ccp_alpha to visualise the pruning trade-off ----
# Plot CV score vs ccp_alpha
alpha_vals = alpha_cv_results["param_model__ccp_alpha"].astype(float).values

mean_col = next(
    c for c in alpha_cv_results.columns
    if c.startswith("mean_test_") and "roc" in c.lower()
)
std_col = next(
    c for c in alpha_cv_results.columns
    if c.startswith("std_test_") and "roc" in c.lower()
)

mean_test_scores = alpha_cv_results[mean_col].astype(float).values
std_test_scores = alpha_cv_results[std_col].astype(float).values

plt.figure(figsize=(10, 6))
plt.errorbar(alpha_vals, mean_test_scores, yerr=std_test_scores, fmt='o-', capsize=5)
plt.axvline(best_ccp_alpha, color='red', linestyle='--', label=f'Best α = {best_ccp_alpha:.4f}')
plt.xlabel('ccp_alpha')
plt.ylabel('Cross-validated ROC-AUC')
plt.title('Cost-Complexity Pruning: Validation Performance vs Alpha')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("report_figures/fig4_pruning_curve_bc.png", dpi=300, bbox_inches="tight")
plt.show()

# 10. Tune Random Forest

Tune a Random Forest with a randomised search over a wide hyperparameter space
(efficient for the large forest grid), refit on ROC-AUC, training-only CV.

In [ ]:
# ---- 10. Tune a Random Forest via randomised search (training-only CV) ----
# Broad space over n_estimators / max_depth / min_samples_split /
# min_samples_leaf / max_features / class_weight; 50 random draws.
rf_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

# Define a broad search space (randomized search first)
param_dist_rf = {
    'model__n_estimators': [100, 300, 500, 700],
    'model__max_depth': [5, 10, 15, 20, None],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['sqrt', 'log2', 0.3, 0.5],
    'model__class_weight': [None, 'balanced']
}

print("Tuning Random Forest with randomized search...")
best_rf_estimator, best_rf_params, rf_cv_results, best_rf_score = tune_model(
    pipeline=rf_pipe,
    param_grid=param_dist_rf,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=SCORING,
    refit='ROC-AUC',
    search_type='random'  # use randomized search for efficiency
)

print("\nBest Random Forest parameters:")
print(best_rf_params)
print(f"Best CV ROC-AUC: {best_rf_score:.4f}")

# 11. Model Comparison (Cross-Validation)

Compare all four models - Dummy, Basic CART, Tuned & Pruned CART and Random
Forest - under identical 5-fold stratified CV folds, collecting mean
discrimination, decision and calibration metrics.

In [ ]:
# ---- 11. Re-evaluate all four models under identical CV folds ----
# Re-instantiate each model and reuse cv_report so the comparison is apples to
# apples. The DataFrame is sorted by CV ROC-AUC.
# Re-instantiate models for comparison (using best selected ones)
models_to_compare = {
    'Dummy Baseline': DummyClassifier(strategy='prior'),
    'Basic CART': DecisionTreeClassifier(criterion='gini', random_state=RANDOM_STATE),
    'Tuned & Pruned CART': pruned_cart,
    'Random Forest': best_rf_estimator
}

# Use your existing cv_report function to get summary and rows
comparison_rows = []
comparison_summaries = {}

for name, model in models_to_compare.items():
    print(f"Evaluating {name}...")
    # We'll use the same SCORING dictionary
    # Note: cv_report expects model, model_name, X, y, cv, scoring, return_predictions
    # We'll call it and append the row
    summary, row = cv_report(model, name, X_train, y_train, cv, SCORING, return_predictions=False)
    comparison_summaries[name] = summary
    comparison_rows.append(row)

# Create comparison DataFrame
comparison_df = pd.DataFrame(comparison_rows)
comparison_df = comparison_df.sort_values('ROC-AUC', ascending=False).round(4)
comparison_df

## 11.1 Model comparison visualisation

Bar chart of the four models across the key cross-validated metrics
(Accuracy, F1, Balanced Accuracy, ROC-AUC, PR-AUC).

In [ ]:
# ---- 11.1 Bar chart of the four models across the key CV metrics ----
plot_metrics = ["Accuracy", "F1", "Balanced Accuracy", "ROC-AUC", "PR-AUC"]

x = np.arange(len(plot_metrics))
width = 0.18
multiplier = 0

fig, ax = plt.subplots(figsize=(14, 8))

for i, row in comparison_df.iterrows():
    offset = width * multiplier
    vals = [row[m] for m in plot_metrics]
    bars = ax.bar(x + offset, vals, width, label=row["Model"])
    ax.bar_label(bars, fmt="%.3f", padding=2, fontsize=8, rotation=0)
    multiplier += 1

ax.set_xlabel("Evaluation Metric", fontsize=12, fontweight="bold")
ax.set_ylabel("Cross-Validated Score", fontsize=12, fontweight="bold")
ax.set_title("Comprehensive Model Comparison: Tuned CART, Random Forest, and Baselines", fontsize=14, fontweight="bold")
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(plot_metrics, fontsize=10)
ax.legend(loc="lower right", fontsize=10, title="Models")
ax.set_ylim(0, 1.15)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.savefig("report_figures/fig5_model_comparison_bc.png", dpi=300, bbox_inches="tight")
plt.show()

## 11.2 Comparison observation

The Random Forest leads on ROC-AUC / PR-AUC; the Tuned & Pruned CART balances
performance with interpretability for the report.

# 12. Select Operating Threshold

Pick the final model (here the best cross-validated model; the Tuned & Pruned
CART may be substituted for interpretability). Use out-of-fold training
predictions to sweep the decision threshold: first enforce a declared
sensitivity target (0.95) and select the highest-specificity threshold that still
meets it; fall back to the Youden-maximising threshold if the target is
infeasible.

In [ ]:
# ---- 12. Sweep the operating threshold on out-of-fold training predictions ----
# Generate OOF probabilities (predict_proba via cross_val_predict so each train
# row is predicted by a fold that never saw it). Then sweep thresholds and
# select the highest-specificity threshold that still meets sensitivity >= 0.95,
# falling back to max Youden if the target is infeasible.
final_model = best_rf_estimator   # or pruned_cart for interpretability

# Generate out-of-fold probabilities on the training set
from sklearn.model_selection import cross_val_predict

oof_proba = cross_val_predict(
    final_model,
    X_train,
    y_train,
    cv=cv,
    method='predict_proba',
    n_jobs=-1
)[:, 1]

# Evaluate thresholds
thresholds = np.linspace(0.01, 0.99, 199)
threshold_metrics = []
for thresh in thresholds:
    pred = (oof_proba >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_train, pred, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1 = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else 0
    threshold_metrics.append({
        'threshold': thresh,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'precision': precision,
        'f1': f1,
        'youden': sensitivity + specificity - 1
    })

threshold_df = pd.DataFrame(threshold_metrics)

# Choose threshold that maximizes Youden's index (or enforce a minimum sensitivity)
# For educational demonstration, we'll choose max Youden
best_youden_idx = threshold_df['youden'].idxmax()
selected_threshold = threshold_df.loc[best_youden_idx, 'threshold']
print(f"Selected threshold (max Youden): {selected_threshold:.3f}")

# Alternatively, enforce a target sensitivity (e.g., 0.95) as per lab manual
TARGET_SENSITIVITY = 0.95
feasible = threshold_df[threshold_df['sensitivity'] >= TARGET_SENSITIVITY]
if not feasible.empty:
    selected_threshold = feasible.sort_values('specificity', ascending=False).iloc[0]['threshold']
    print(f"Selected threshold (target sensitivity >= {TARGET_SENSITIVITY}): {selected_threshold:.3f}")
else:
    print("No threshold meets the target sensitivity; using max Youden.")

# Plot threshold curves
plt.figure(figsize=(10, 6))
plt.plot(threshold_df['threshold'], threshold_df['sensitivity'], label='Sensitivity', linewidth=2)
plt.plot(threshold_df['threshold'], threshold_df['specificity'], label='Specificity', linewidth=2)
plt.axvline(selected_threshold, color='red', linestyle='--', label=f'Selected threshold = {selected_threshold:.2f}')
plt.xlabel('Threshold')
plt.ylabel('Metric')
plt.title('Sensitivity and Specificity vs Threshold (Out-of-Fold)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("report_figures/fig6_threshold_bc.png", dpi=300, bbox_inches="tight")
plt.show()

# 13. Final Evaluation on the Locked Test Set

Lock the model on all training data and evaluate on the test set exactly once.
Report the full metric block (sensitivity, specificity, precision, NPV, F1,
balanced accuracy, ROC-AUC, PR-AUC, Brier, MCC) and the explicit TN/FP/FN/TP
counts at the chosen threshold.

In [ ]:
# ---- 13. Lock the final model on all train data; score the test set ONCE ----
# `evaluate_binary` builds the full metric block used in the report.
# Fit the final model on the entire training set (with best parameters)
final_model.fit(X_train, y_train)

# Predict probabilities on test set
test_proba = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= selected_threshold).astype(int)

# Compute metrics using a function
def evaluate_binary(y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    metrics = {
        'threshold': selected_threshold,
        'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp,
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'sensitivity': recall_score(y_true, y_pred, zero_division=0),
        'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0,
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'npv': tn / (tn + fn) if (tn + fn) > 0 else 0,
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'mcc': matthews_corrcoef(y_true, y_pred),
        'roc_auc': roc_auc_score(y_true, y_proba),
        'pr_auc': average_precision_score(y_true, y_proba),
        'brier_score': brier_score_loss(y_true, y_proba)
    }
    return metrics

test_metrics = evaluate_binary(y_test, test_pred, test_proba)
test_metrics_df = pd.DataFrame([test_metrics]).round(4)
test_metrics_df

In [ ]:
# ---- 13. Plot test-set confusion matrix, ROC curve and PR curve (saved) ----
# Confusion Matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, test_pred,
    display_labels=['Benign', 'Malignant'],
    cmap='Blues',
    values_format='d'
)
plt.title(f'Test Confusion Matrix (threshold = {selected_threshold:.2f})')
plt.show()

# ROC Curve
RocCurveDisplay.from_predictions(y_test, test_proba, name='Final Model')
plt.plot([0,1], [0,1], 'k--', label='Chance')
plt.legend()
plt.title('ROC Curve - Test Set')
plt.show()

# PR Curve
PrecisionRecallDisplay.from_predictions(y_test, test_proba, name='Final Model')
plt.axhline(y_test.mean(), color='gray', linestyle='--', label='Positive prevalence')
plt.legend()
plt.title('Precision-Recall Curve - Test Set')
plt.savefig("report_figures/fig7_confusion_roc_pr_bc.png", dpi=300, bbox_inches="tight")
plt.show()

# 14. Calibration and Save Artifacts

Assess probability calibration of the final model on the test set, then persist
every artefact the CLI/GUI need: the fitted final pipeline, the operating
threshold, individual model pipelines, the CV and test results CSVs, and the
metadata JSON - written via the shared `meddiag_common.save_artifacts` engine so
the notebook and the CLI can never disagree.

## 14.1 Calibration plot (reliability diagram)

Evaluate probability calibration of the final model on the test set. A perfectly
calibrated model lies on the diagonal; the Brier score summarises the
reliability gap.

In [ ]:
# ---- 14.1 Calibration plot (reliability diagram) on the test set ----
from sklearn.calibration import calibration_curve

def plot_calibration(y_true, prob, model_name, ax, color):
    prob_true, prob_pred = calibration_curve(y_true, prob, n_bins=10, strategy="uniform")
    ax.plot(prob_pred, prob_true, marker="o", linewidth=2, label=model_name, color=color)

fig, ax = plt.subplots(figsize=(9, 8))
plot_calibration(y_test, test_proba, f"Final Model (Brier={test_metrics['brier_score']:.3f})", ax, "blue")
ax.plot([0, 1], [0, 1], "k--", label="Perfect Calibration", alpha=0.6)
ax.set_xlabel("Mean Predicted Probability")
ax.set_ylabel("Observed Fraction of Positives")
ax.set_title("Calibration Plot (Reliability Diagram)")
ax.legend(loc="upper left", fontsize=8)
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ---- 14. Persist artefacts via the shared meddiag_common engine ----
# Build a minimal `state` dict from the objects already trained in this notebook
# and call `meddiag_common.save_artifacts` so the CLI/GUI can load the saved
# pipeline, threshold and metadata from `artifacts/breast_*` without retraining.
# A legacy `lab02_full_artifact_bundle.joblib` bundle is also preserved for
# backwards compatibility with earlier versions of the notebook.
import joblib, platform, sklearn  # noqa: E402
from datetime import datetime       # noqa: E402

# Map the notebook's trained estimators into the shared-registry naming.
_models_registry = {
    "Dummy (prior)": build_pipeline(DummyClassifier(strategy="prior")),
    "Basic CART": basic_cart,
    "Tuned and pruned CART": pruned_cart,
    "Random Forest": best_rf_estimator,
    "Logistic Regression": build_pipeline(
        __import__("sklearn.linear_model", fromlist=["LogisticRegression"]).LogisticRegression(
            max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE
        )
    ),
}
# Fit any unfitted pipeline in the registry on the full train split so that
# every saved .joblib is ready-to-predict, matching the CLI's expectations.
for _name, _pipe in _models_registry.items():
    _inner = _pipe.named_steps.get("classifier") if hasattr(_pipe, "named_steps") else _pipe
    if not getattr(_inner, "classes_", None):
        _pipe.fit(X_train, y_train)

_state = {
    "spec": M.spec("breast"),
    "X": dataset.drop(columns=["Diagnosis"]), "y": y,
    "X_train": X_train, "X_test": X_test, "y_train": y_train, "y_test": y_test,
    "cv": cv,
    "models": _models_registry,
    "pruned_gs1": None, "pruned_gs2": None,
    "pruned_params": {f"model__{k}": v for k, v in best_cart_params.items()
                     if k != "model__ccp_alpha"} if "best_cart_params" in dir() else {},
    "rf_gs": None, "rf_params": best_rf_params if "best_rf_params" in dir() else {},
    "cv_table": comparison_df.rename(columns={
        "Accuracy": "accuracy", "Balanced Accuracy": "balanced_accuracy",
        "Precision": "precision", "Recall": "recall", "F1": "f1",
        "ROC-AUC": "roc_auc", "PR-AUC": "pr_auc",
    }) if "comparison_df" in dir() else pd.DataFrame(
        [{"Model": name, "roc_auc_mean": 0} for name in _models_registry]),
    "oof_prob": None, "threshold_df": threshold_df if "threshold_df" in dir() else pd.DataFrame(),
    "threshold": float(selected_threshold),
    "threshold_target_satisfied": float(
        threshold_df.loc[threshold_df.threshold <= selected_threshold, "sensitivity"].max()
    ) if "threshold_df" in dir() and len(threshold_df) else 0.95,
    "best_name": "Random Forest" if final_model is best_rf_estimator else "Tuned and pruned CART",
    "best_pipe": final_model,
    "test_metrics": {**test_metrics,
                     **{"brier": test_metrics.get("brier_score",
                                                  test_metrics.get("brier", 0.0))}},
    "default_threshold_metrics": {**test_metrics},
    "all_test_rows": [dict(Model="Random Forest" if final_model is best_rf_estimator
                           else "Tuned and pruned CART", **test_metrics)],
    "test_prob": test_proba, "test_pred": test_pred,
    "feature_cols": list(X.columns), "class_names": ("Benign", "Malignant"),
    "numeric_cols": list(X.columns), "categorical_cols": [],
    "complexity": {},
}

_paths = M.save_artifacts("breast", _state, tag="breast")
# Also keep the legacy bundle for backwards compatibility.
_legacy_dir = Path("artifacts"); _legacy_dir.mkdir(exist_ok=True)
joblib.dump({k: v for k, v in {
    "preprocessor": preprocessor, "dummy_pipeline": dummy_pipeline,
    "basic_cart_pipeline": basic_cart, "best_cart_pipeline": best_cart_estimator,
    "pruned_cart_pipeline": pruned_cart, "best_random_forest_pipeline": best_rf_estimator,
    "final_model": final_model, "threshold": selected_threshold,
    "positive_class": "Malignant", "feature_names": list(X_train.columns),
    "random_state": RANDOM_STATE, "test_metrics": test_metrics,
    "software": {"python": platform.python_version(),
                 "scikit_learn": sklearn.__version__,
                 "pandas": pd.__version__, "numpy": np.__version__},
}.items() if not isinstance(v, (dict, list, str, int, float, bool)) or v is None},
    _legacy_dir / "lab02_full_artifact_bundle.joblib")

print("Artifacts saved (CLI/GUI compatible) to:", os.path.abspath(M.ART_DIR))
print("Metadata:", _paths.get("metadata"))
from pathlib import Path
import joblib
import platform
import sklearn

artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(exist_ok=True)

artifact_bundle = {
    "preprocessor": preprocessor,
    "dummy_pipeline": dummy_pipeline,
    "basic_cart_pipeline": basic_cart,
    "cart_tuning_pipeline": cart_pipe,
    "best_cart_pipeline": best_cart_estimator,
    "pruning_pipeline": prune_pipe,
    "pruned_cart_pipeline": pruned_cart,
    "random_forest_pipeline": rf_pipe,
    "best_random_forest_pipeline": best_rf_estimator,
    "final_model": final_model,
    "threshold": selected_threshold,
    "positive_class": "malignant",
    "feature_names": list(X_train.columns),
    "random_state": RANDOM_STATE,
    "test_metrics": test_metrics,
    "software": {
        "python": platform.python_version(),
        "scikit_learn": sklearn.__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__
    },
    "intended_use": "Educational predictive-analytics laboratory only",
    "prohibited_use": "Clinical diagnosis, treatment, triage, or patient management"
}

joblib.dump(artifact_bundle, artifacts_dir / "lab02_full_artifact_bundle.joblib")

# Save each major fitted object separately as well
for name, obj in artifact_bundle.items():
    if isinstance(obj, (dict, list, str, int, float, bool)):
        continue
    joblib.dump(obj, artifacts_dir / f"{name}.joblib")

pd.DataFrame([test_metrics]).to_csv(artifacts_dir / "lab02_test_metrics.csv", index=False)
print("Artifacts saved to:", artifacts_dir)
print("Saved bundle:", artifacts_dir / "lab02_full_artifact_bundle.joblib")